# 04 — Tree-based Models + Optuna Tuning

**Mode**: fast — subsample 200k baris, model lebih ringan, Optuna 20 trials.
Untuk hasil maksimal, naikkan `SUBSAMPLE` dan `N_TRIALS` di cell konfigurasi.

In [1]:
import sys, pathlib, time
sys.path.append(str(pathlib.Path.cwd().parent))
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, joblib, mlflow, optuna
from tqdm import tqdm
optuna.logging.set_verbosity(optuna.logging.WARNING)
from src.data import load_processed
from src.evaluate import regression_metrics
from src.config import MLFLOW_URI, EXPERIMENT_NAME, MODELS_DIR
mlflow.set_tracking_uri(MLFLOW_URI); mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location=('file:///D:/ZE5 PORTOFOLIO DS/Q-Factor Prediction in Optical Communication '
 'Systems/mlruns/622530031528693603'), creation_time=1777606059837, experiment_id='622530031528693603', last_update_time=1777606059837, lifecycle_stage='active', name='qfactor_prediction', tags={}, trace_location=None, workspace='default'>

In [2]:
# === Konfigurasi cepat ===
SUBSAMPLE = 200_000   # naikkan ke None untuk full data
N_TRIALS = 20         # naikkan ke 50-100 untuk hasil lebih baik
SEED = 42

In [3]:
X_train_full, y_train_full = load_processed('train')
X_val, y_val = load_processed('val')
if SUBSAMPLE and SUBSAMPLE < len(X_train_full):
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X_train_full), SUBSAMPLE, replace=False)
    X_train, y_train = X_train_full[idx], y_train_full[idx]
else:
    X_train, y_train = X_train_full, y_train_full
print('train:', X_train.shape, ' val:', X_val.shape)

train: (200000, 5)  val: (150000, 5)


## 1. Random Forest (ringan: 100 pohon, 50k subsample)

In [4]:
from sklearn.ensemble import RandomForestRegressor
rf_idx = np.random.default_rng(SEED).choice(len(X_train), min(50_000, len(X_train)), replace=False)
t0 = time.time()
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED, verbose=1)
rf.fit(X_train[rf_idx], y_train[rf_idx])
m = regression_metrics(y_val, rf.predict(X_val)); print(f'RF -> {m}  ({time.time()-t0:.1f}s)')
with mlflow.start_run(run_name='RandomForest'):
    mlflow.log_params({'model':'RF','n_estimators':100}); mlflow.log_metrics(m)
joblib.dump(rf, MODELS_DIR / 'RandomForest.joblib')
rf_metrics = m

RF -> {'RMSE': 0.13654997714005188, 'MAE': 0.1081684807424863, 'R2': 0.986288057992547, 'MAPE': 2.155366211408864}  (6.0s)


## 2. XGBoost (early stopping, log per 25 iterasi)

In [5]:
from xgboost import XGBRegressor
t0 = time.time()
xgb = XGBRegressor(
    n_estimators=400, learning_rate=0.08, max_depth=8,
    subsample=0.9, colsample_bytree=0.9, tree_method='hist',
    random_state=SEED, n_jobs=-1, early_stopping_rounds=20,
)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=25)
m = regression_metrics(y_val, xgb.predict(X_val)); print(f'XGB -> {m}  ({time.time()-t0:.1f}s)')
with mlflow.start_run(run_name='XGBoost'):
    mlflow.log_params({'model':'XGB'}); mlflow.log_metrics(m)
joblib.dump(xgb, MODELS_DIR / 'XGBoost.joblib')
xgb_metrics = m

[0]	validation_0-rmse:1.08918
[25]	validation_0-rmse:0.27727
[50]	validation_0-rmse:0.12404
[75]	validation_0-rmse:0.10771
[100]	validation_0-rmse:0.10608
[125]	validation_0-rmse:0.10582
[150]	validation_0-rmse:0.10572
[175]	validation_0-rmse:0.10566
[200]	validation_0-rmse:0.10561
[225]	validation_0-rmse:0.10558
[250]	validation_0-rmse:0.10557
[275]	validation_0-rmse:0.10558
[282]	validation_0-rmse:0.10558
XGB -> {'RMSE': 0.10556906327142476, 'MAE': 0.0840839147567749, 'R2': 0.9918042421340942, 'MAPE': 1.6894152164459229}  (5.6s)


## 3. LightGBM (early stopping, log per 25 iterasi)

In [6]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
t0 = time.time()
lgbm = LGBMRegressor(
    n_estimators=500, learning_rate=0.08, num_leaves=63,
    subsample=0.9, colsample_bytree=0.9, random_state=SEED, n_jobs=-1, verbose=-1,
)
lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[early_stopping(20), log_evaluation(25)])
m = regression_metrics(y_val, lgbm.predict(X_val)); print(f'LGBM -> {m}  ({time.time()-t0:.1f}s)')
with mlflow.start_run(run_name='LightGBM'):
    mlflow.log_params({'model':'LGBM'}); mlflow.log_metrics(m)
joblib.dump(lgbm, MODELS_DIR / 'LightGBM.joblib')
lgbm_metrics = m

Training until validation scores don't improve for 20 rounds
[25]	valid_0's l2: 0.0646458
[50]	valid_0's l2: 0.0159186
[75]	valid_0's l2: 0.0119264
[100]	valid_0's l2: 0.0114688
[125]	valid_0's l2: 0.0113464
[150]	valid_0's l2: 0.011309
[175]	valid_0's l2: 0.0112803
[200]	valid_0's l2: 0.0112466
[225]	valid_0's l2: 0.0112134
[250]	valid_0's l2: 0.0111927
[275]	valid_0's l2: 0.0111656
[300]	valid_0's l2: 0.0111468
[325]	valid_0's l2: 0.0111303
[350]	valid_0's l2: 0.0111212
[375]	valid_0's l2: 0.0111146
[400]	valid_0's l2: 0.0111071
[425]	valid_0's l2: 0.0110969
[450]	valid_0's l2: 0.0110905
[475]	valid_0's l2: 0.0110871
[500]	valid_0's l2: 0.0110835
Did not meet early stopping. Best iteration is:
[496]	valid_0's l2: 0.0110827


LGBM -> {'RMSE': 0.10527457492794298, 'MAE': 0.08389781614252036, 'R2': 0.9918499045069751, 'MAPE': 1.6847353465903037}  (3.6s)


## 4. CatBoost (log per 50 iterasi)

In [7]:
from catboost import CatBoostRegressor
t0 = time.time()
cb = CatBoostRegressor(iterations=500, learning_rate=0.08, depth=8, random_seed=SEED, verbose=50, early_stopping_rounds=20)
cb.fit(X_train, y_train, eval_set=(X_val, y_val))
m = regression_metrics(y_val, cb.predict(X_val)); print(f'CatBoost -> {m}  ({time.time()-t0:.1f}s)')
with mlflow.start_run(run_name='CatBoost'):
    mlflow.log_params({'model':'CatBoost'}); mlflow.log_metrics(m)
joblib.dump(cb, MODELS_DIR / 'CatBoost.joblib')
cb_metrics = m

0:	learn: 1.0838837	test: 1.0844200	best: 1.0844200 (0)	total: 173ms	remaining: 1m 26s
50:	learn: 0.1224983	test: 0.1220171	best: 0.1220171 (50)	total: 1.39s	remaining: 12.2s
100:	learn: 0.1042483	test: 0.1040377	best: 0.1040377 (100)	total: 2.58s	remaining: 10.2s
150:	learn: 0.1028002	test: 0.1026877	best: 0.1026877 (150)	total: 3.81s	remaining: 8.82s
200:	learn: 0.1019191	test: 0.1019048	best: 0.1019048 (200)	total: 4.99s	remaining: 7.42s
250:	learn: 0.1014272	test: 0.1015145	best: 0.1015145 (250)	total: 6.16s	remaining: 6.11s
300:	learn: 0.1010681	test: 0.1012526	best: 0.1012526 (300)	total: 7.43s	remaining: 4.91s
350:	learn: 0.1007778	test: 0.1011004	best: 0.1011004 (350)	total: 8.83s	remaining: 3.75s
400:	learn: 0.1005143	test: 0.1009626	best: 0.1009626 (400)	total: 10.5s	remaining: 2.59s
450:	learn: 0.1002880	test: 0.1008860	best: 0.1008860 (450)	total: 12.1s	remaining: 1.31s
499:	learn: 0.1000740	test: 0.1008223	best: 0.1008223 (499)	total: 13.6s	remaining: 0us

bestTest = 0.100

In [8]:
pd.DataFrame([
    dict(model='RandomForest', **rf_metrics),
    dict(model='XGBoost', **xgb_metrics),
    dict(model='LightGBM', **lgbm_metrics),
    dict(model='CatBoost', **cb_metrics),
]).set_index('model').sort_values('RMSE')

,RMSE,MAE,R2,MAPE
model,,,,
CatBoost,0.100822,0.080426,0.992525,1.617631
LightGBM,0.105275,0.083898,0.991850,1.684735
XGBoost,0.105569,0.084084,0.991804,1.689415
RandomForest,0.136550,0.108168,0.986288,2.155366


## 5. Optuna tuning XGBoost (20 trials)

In [9]:
from sklearn.metrics import mean_squared_error
tune_idx = np.random.default_rng(SEED).choice(len(X_train), min(80_000, len(X_train)), replace=False)
Xt, yt = X_train[tune_idx], y_train[tune_idx]

def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 600),
        learning_rate=trial.suggest_float('learning_rate', 1e-2, 0.2, log=True),
        max_depth=trial.suggest_int('max_depth', 4, 10),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-6, 1.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-6, 1.0, log=True),
    )
    m = XGBRegressor(tree_method='hist', random_state=SEED, n_jobs=-1, early_stopping_rounds=15, **params)
    m.fit(Xt, yt, eval_set=[(X_val, y_val)], verbose=False)
    return float(np.sqrt(mean_squared_error(y_val, m.predict(X_val))))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
pbar = tqdm(total=N_TRIALS, desc='optuna trials')
study.optimize(objective, n_trials=N_TRIALS, callbacks=[lambda s, t: pbar.update(1)])
pbar.close()
print('best RMSE:', study.best_value)
study.best_params

best RMSE: 0.10364078656406425


{'n_estimators': 597,
 'learning_rate': 0.0359341984611382,
 'max_depth': 5,
 'subsample': 0.9908983403146366,
 'colsample_bytree': 0.6978753170579768,
 'reg_alpha': 0.0003069158766735034,
 'reg_lambda': 0.00014425793182034382}

In [10]:
best = XGBRegressor(tree_method='hist', random_state=SEED, n_jobs=-1, **study.best_params)
best.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
m = regression_metrics(y_val, best.predict(X_val)); print('XGBoost_tuned ->', m)
with mlflow.start_run(run_name='XGBoost_tuned'):
    mlflow.log_params(study.best_params); mlflow.log_metrics(m)
joblib.dump(best, MODELS_DIR / 'XGBoost_tuned.joblib')

[0]	validation_0-rmse:1.15433
[50]	validation_0-rmse:0.46448
[100]	validation_0-rmse:0.20775
[150]	validation_0-rmse:0.13066
[200]	validation_0-rmse:0.10992
[250]	validation_0-rmse:0.10461
[300]	validation_0-rmse:0.10318
[350]	validation_0-rmse:0.10275
[400]	validation_0-rmse:0.10257
[450]	validation_0-rmse:0.10249
[500]	validation_0-rmse:0.10244
[550]	validation_0-rmse:0.10239
[596]	validation_0-rmse:0.10234
XGBoost_tuned -> {'RMSE': 0.10234336798356372, 'MAE': 0.08160831034183502, 'R2': 0.9922974109649658, 'MAPE': 1.640235424041748}


['D:\\ZE5 PORTOFOLIO DS\\Q-Factor Prediction in Optical Communication Systems\\models\\XGBoost_tuned.joblib']